# 🩺 MediVLM: Qwen2.5-3B + DistilBERT on MIMIC-CXR Sample

This Colab Notebook implements a brand new Vision-Language architecture for MediVLM:
1. **Image Encoder**: Faster R-CNN (top 8 patches) + CLIP-ViT
2. **Text Encoder (Training Only)**: DistilBERT (`distilbert-base-uncased`)
3. **Report Decoder**: Qwen2.5-3B-Instruct with **LoRA** (Low-Rank Adaptation)

The notebook automates environment setup, mounts Google Drive to extract the MIMIC-CXR zip, generates annotations, defines the custom model, and runs an integrated training and inference loop!

## 1. Setup Environment
Clone the repository and install all required python dependencies, including `peft` and `accelerate` for Qwen and LoRA.

In [ ]:
!git clone https://github.com/sonai-commits/MediVLM.git
%cd MediVLM
!pip install -r requirements.txt
!pip install radgraph RaTEScore peft accelerate transformers bitsandbytes "torchao>=0.16.0"
!pip install -e .

## 2. Mount Google Drive and Unzip Dataset
Mount your Drive and extract the files directly into the correct folder.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# CHANGE THIS if your zip file is named differently or is in a subfolder!
ZIP_PATH = '/content/drive/MyDrive/mimic_cxr.zip'

os.makedirs('data/mimic_cxr', exist_ok=True)

if os.path.exists(ZIP_PATH):
    print(f"\nFound {ZIP_PATH}! Unzipping into data/mimic_cxr...")
    !unzip -q -n "{ZIP_PATH}" -d data/mimic_cxr
    print("Unzip complete!")
else:
    print(f"Error: Could not find {ZIP_PATH}. Please check your Drive path.")

## 3. Generate Annotations from CSV
The Kaggle dataset doesn't have the `annotations.json` file. We will generate it from the `mimic_cxr_aug_train.csv` file.

In [ ]:
import pandas as pd
import json
import ast
import os

print("Reading Kaggle CSVs...")
train_df = pd.read_csv('data/mimic_cxr/mimic_cxr_aug_train.csv')
val_df = pd.read_csv('data/mimic_cxr/mimic_cxr_aug_validate.csv')

def process_df(df):
    samples = []
    for idx, row in df.iterrows():
        try:
            img_list = ast.literal_eval(row['image'])
            # Prepend the folder structure so the dataloader can find it
            corrected_paths = [f"official_data_iccv_final/{p}" for p in img_list]
            samples.append({
                "id": str(row['subject_id']),
                "image_path": corrected_paths,
                "report": str(row['text'])
            })
        except Exception as e:
            continue
    return samples

train_samples = process_df(train_df)
val_samples = process_df(val_df)

annotations = {
    "train": train_samples,
    "val": val_samples,
    "test": val_samples
}

with open('data/mimic_cxr/annotations.json', 'w') as f:
    json.dump(annotations, f)
    
print(f"\nSaved annotations.json with {len(train_samples)} training samples.")

## 4. Filter Existing Images & Create Sample
Since the zip file you uploaded only contains a subset of all the images listed in the CSV, we must filter out missing images and create `annotations_sample.json`.

In [ ]:
print("Filtering missing images to create annotations_sample.json...")

with open('data/mimic_cxr/annotations.json', 'r') as f:
    ann = json.load(f)

def get_existing(samples, limit):
    valid = []
    for s in samples:
        path = "data/mimic_cxr/" + s['image_path'][0]
        if os.path.exists(path):
            valid.append(s)
            if limit is not None and len(valid) == limit:
                break
    return valid

sample_data = {
    "train": get_existing(ann['train'], None),
    "val": get_existing(ann['val'], None),
    "test": get_existing(ann['test'], None)
}

with open('data/mimic_cxr/annotations_sample.json', 'w') as f:
    json.dump(sample_data, f, indent=4)

print(f"Success! Saved annotations_sample.json with {len(sample_data['train'])} valid training images.")

## 5. Create Configuration File
We will generate a custom config specifically for the Qwen architecture, using a smaller batch size (e.g. 2 or 4) to ensure it fits in Colab's 15 GB VRAM.

In [ ]:
%%writefile configs/mimic_cxr_qwen_sample.yaml
detector:
  num_anatomical_classes: 29
  weights_path: null
  top_p_patches: 8
  patch_size: 28
  freeze: true
image_encoder:
  model_name: openai/clip-vit-large-patch14
  image_size: 224
  freeze: true
text_encoder:
  model_name: distilbert-base-uncased
  max_length: 128
  freeze: true
projection:
  proj_dim: 512
  temperature: 0.07
  dropout: 0.1
fusion:
  num_heads: 8
  dropout: 0.1
decoder:
  model_name: Qwen/Qwen2.5-3B-Instruct
  trainable_blocks: 0
  max_length: 150
  beam_size: 4
data:
  dataset: mimic_cxr
  root: data/mimic_cxr
  ann_file: annotations_sample.json
  image_size: 224
  batch_size: 2
  num_workers: 0
training:
  epochs: 1
  lr: 2.0e-5
  lambda_ce: 1.0
  lambda_contrast: 0.7
  output_dir: outputs/mimic_cxr_qwen_sample
  mixed_precision: true

## 6. Define Custom Architecture Components
Here we implement the custom `DistilBertTextEncoder`, `QwenReportDecoder` with LoRA, and the wrapping `MediVLMLoRA` model.

In [ ]:
import torch
import torch.nn as nn
from typing import List, Optional, Dict
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
from medivlm.models.detector import FasterRCNNDetector, DetectorOutput
from medivlm.models.image_encoder import CLIPImageEncoder
from medivlm.models.projection import ProjectionHead, contrastive_loss
from medivlm.models.fusion import CrossAttentionFusion
from medivlm.utils.config import MediVLMConfig
from medivlm.models.medivlm import MediVLMForwardOutput

class DistilBertTextEncoder(nn.Module):
    def __init__(
        self,
        model_name: str = "distilbert-base-uncased",
        freeze: bool = True,
        max_length: int = 128,
    ) -> None:
        super().__init__()
        self.max_length = max_length
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.bert = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.bert.config.hidden_size
        self._frozen = bool(freeze)

        if freeze:
            for p in self.bert.parameters():
                p.requires_grad_(False)
            self.bert.eval()

    def train(self, mode: bool = True):
        super().train(mode)
        if getattr(self, "_frozen", False):
            self.bert.eval()
        return self

    def forward(self, texts: Optional[List[str]] = None) -> Optional[dict]:
        if texts is None:
            return None
        tok = self.tokenizer(
            texts, padding="max_length", truncation=True, 
            max_length=self.max_length, return_tensors="pt"
        )
        device = next(self.bert.parameters()).device
        input_ids = tok["input_ids"].to(device)
        attention_mask = tok["attention_mask"].to(device)

        with torch.set_grad_enabled(self.training and not self._frozen):
            out = self.bert(input_ids=input_ids, attention_mask=attention_mask, return_dict=True)

        token_embeds = out.last_hidden_state
        pooled = token_embeds[:, 0]  # Take CLS token for pooling

        return {
            "token_embeds": token_embeds,
            "pooled": pooled,
            "attention_mask": attention_mask,
            "input_ids": input_ids,
        }

class QwenReportDecoder(nn.Module):
    def __init__(
        self,
        model_name: str = "Qwen/Qwen2.5-3B-Instruct",
        fusion_dim: int = 512,
        max_length: int = 150,
        beam_size: int = 4,
    ) -> None:
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        device_map = "auto" if torch.cuda.is_available() else None
        # Load in half-precision to save memory
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=dtype
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, torch_dtype=dtype, device_map=device_map, quantization_config=bnb_config
        )
        self.model.config.pad_token_id = self.tokenizer.pad_token_id

        # Wrap with LoRA
        lora_config = LoraConfig(
            r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
            lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
        )
        self.model = get_peft_model(self.model, lora_config)
        self.model.print_trainable_parameters()

        self.hidden_size = self.model.config.hidden_size
        self.fusion_proj = nn.Linear(fusion_dim, self.hidden_size)

        self.max_length = max_length
        self.beam_size = beam_size

    def _build_inputs_embeds(
        self, fusion: torch.Tensor, input_ids: Optional[torch.Tensor]
    ) -> Dict[str, torch.Tensor]:
        B = fusion.size(0)
        embed_layer = self.model.get_input_embeddings()
        dtype = embed_layer.weight.dtype
        prefix = self.fusion_proj(fusion).to(dtype)

        if input_ids is None:
            token_embeds = prefix.new_zeros((B, 0, self.hidden_size))
            token_mask = torch.ones(B, 0, device=prefix.device, dtype=torch.long)
        else:
            token_embeds = embed_layer(input_ids)
            token_mask = (input_ids != self.tokenizer.pad_token_id).long()

        prefix_mask = torch.ones(B, prefix.size(1), device=prefix.device, dtype=torch.long)
        inputs_embeds = torch.cat([prefix, token_embeds], dim=1)
        attention_mask = torch.cat([prefix_mask, token_mask], dim=1)
        return {"inputs_embeds": inputs_embeds, "attention_mask": attention_mask, "prefix_len": prefix.size(1)}

    def forward(self, fusion: torch.Tensor, labels_input_ids: torch.Tensor) -> Dict[str, torch.Tensor]:
        built = self._build_inputs_embeds(fusion, labels_input_ids)
        inputs_embeds = built["inputs_embeds"]
        attention_mask = built["attention_mask"]
        prefix_len = built["prefix_len"]

        labels = labels_input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        pad_labels = labels.new_full((labels.size(0), prefix_len), -100)
        full_labels = torch.cat([pad_labels, labels], dim=1)

        out = self.model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=full_labels,
            return_dict=True,
        )
        return {"loss": out.loss, "logits": out.logits}

    @torch.no_grad()
    def generate(self, fusion: torch.Tensor, prompt: Optional[List[str]] = None, max_length: int = None, num_beams: int = None) -> List[str]:
        device = fusion.device
        prompt = prompt or [""] * fusion.size(0)
        enc = self.tokenizer(prompt, return_tensors="pt", padding=True).to(device)
        input_ids = enc["input_ids"] if enc["input_ids"].numel() > 0 else None
        
        built = self._build_inputs_embeds(fusion, input_ids)
        
        out = self.model.generate(
            inputs_embeds=built["inputs_embeds"],
            attention_mask=built["attention_mask"],
            max_new_tokens=max_length or self.max_length,
            num_beams=num_beams or self.beam_size,
            pad_token_id=self.tokenizer.pad_token_id,
            eos_token_id=self.tokenizer.eos_token_id,
        )
        return self.tokenizer.batch_decode(out, skip_special_tokens=True)

class MediVLMLoRA(nn.Module):
    def __init__(self, config: MediVLMConfig) -> None:
        super().__init__()
        self.config = config
        
        self.detector = FasterRCNNDetector(
            num_anatomical_classes=config.detector.num_anatomical_classes,
            weights_path=config.detector.weights_path,
            top_p_patches=config.detector.top_p_patches,
            freeze=config.detector.freeze,
        )
        
        self.image_encoder = CLIPImageEncoder(
            model_name=config.image_encoder.model_name,
            freeze=config.image_encoder.freeze,
        )
        
        self.text_encoder = DistilBertTextEncoder(
            model_name=config.text_encoder.model_name,
            freeze=config.text_encoder.freeze,
        )
        
        proj_dim = config.projection.proj_dim
        self.proj_visual = ProjectionHead(self.image_encoder.hidden_size, proj_dim)
        self.proj_text = ProjectionHead(self.text_encoder.hidden_size, proj_dim)
        self.temperature = config.projection.temperature
        
        self.fusion = CrossAttentionFusion(dim=proj_dim, num_heads=config.fusion.num_heads)
        
        self.decoder = QwenReportDecoder(
            model_name=config.decoder.model_name,
            fusion_dim=proj_dim,
            max_length=config.decoder.max_length,
            beam_size=config.decoder.beam_size,
        )
        
        self.lambda_ce = config.training.lambda_ce
        self.lambda_contrast = config.training.lambda_contrast

    def forward(self, images: torch.Tensor, reports: Optional[List[str]] = None) -> MediVLMForwardOutput:
        detections = self.detector(images)
        patches = torch.stack([d.patches for d in detections], dim=0)
        coords = torch.stack([d.norm_coords for d in detections], dim=0)
        
        visual = self.image_encoder(patches.to(images.device), coords.to(images.device))
        
        # Text encoding happens only during training
        enc_txt = self.text_encoder(reports) if self.training and reports is not None else None
        
        v_proj_tokens = self.proj_visual(visual)
        v_proj_pool = v_proj_tokens.mean(dim=1)
        
        t_proj_tokens, t_proj_pool, text_mask = None, None, None
        if enc_txt is not None:
            t_proj_tokens = self.proj_text(enc_txt["token_embeds"])
            t_proj_pool = self.proj_text(enc_txt["pooled"])
            text_mask = enc_txt["attention_mask"]
            
        fused = self.fusion(v_proj_tokens, t_proj_tokens, text_mask)
        
        loss_ce, loss_contrast, loss, logits = None, None, None, None
        
        if reports is not None:
            enc = self.decoder.tokenizer(
                reports, return_tensors="pt", padding="max_length", 
                truncation=True, max_length=self.decoder.max_length
            ).to(images.device)
            dec_out = self.decoder(fused, enc["input_ids"])
            loss_ce = dec_out["loss"]
            logits = dec_out["logits"]
            
            # Use same precision type for contrastive loss addition
            loss_contrast = contrastive_loss(v_proj_pool, t_proj_pool, temperature=self.temperature).to(loss_ce.dtype)
            loss = self.lambda_ce * loss_ce + self.lambda_contrast * loss_contrast
            
        return MediVLMForwardOutput(
            loss=loss, loss_ce=loss_ce, loss_contrast=loss_contrast, 
            logits=logits, fusion=fused, visual_tokens=v_proj_tokens, 
            text_tokens=t_proj_tokens, detections=detections
        )

    @torch.no_grad()
    def generate(self, images: torch.Tensor, num_beams: int = 4) -> List[str]:
        detections = self.detector(images)
        patches = torch.stack([d.patches for d in detections], dim=0)
        coords = torch.stack([d.norm_coords for d in detections], dim=0)
        
        visual = self.image_encoder(patches.to(images.device), coords.to(images.device))
        v_tokens = self.proj_visual(visual)
        fused = self.fusion(v_tokens, None, None)
        return self.decoder.generate(fused, num_beams=num_beams)


## 7. Run the Fast Sample Training
Run a custom PyTorch training loop on the Qwen architecture using the loaded dataset!

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import torch
from medivlm.data import build_dataloader
from medivlm.utils import load_config
from tqdm.auto import tqdm
from pathlib import Path

# Load config
cfg = load_config("configs/mimic_cxr_qwen_sample.yaml")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Building data loaders...")
train_loader = build_dataloader(cfg.data, "train")

print("Initializing MediVLMLoRA (Qwen2.5-3B + DistilBERT + CLIP)...")
model = MediVLMLoRA(cfg).to(device)

# Prepare optimizer
trainable = [p for p in model.parameters() if p.requires_grad]
print(f"Trainable parameters: {sum(p.numel() for p in trainable):,}")
optimizer = torch.optim.AdamW(trainable, lr=cfg.training.lr)
# Use the modern torch.amp API (torch.cuda.amp is deprecated)
scaler = torch.amp.GradScaler('cuda', enabled=True)

# Training loop
epochs = cfg.training.epochs
model.train()

for epoch in range(1, epochs + 1):
    total_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}")
    
    for batch in pbar:
        images = batch["images"].to(device)
        reports = batch["reports"]
        
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            out = model(images, reports=reports)
            loss = out.loss
            
        if loss is None:
            continue
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.4f}", "ce": f"{out.loss_ce.item():.4f}"})
        
    print(f"Epoch {epoch} finished | Avg Loss: {total_loss / len(train_loader):.4f}")
    # Free up unused VRAM at the end of each epoch
    torch.cuda.empty_cache()

# Save the checkpoint
output_dir = Path(cfg.training.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), output_dir / "qwen_lora_best.pt")
print("Training Complete! Weights saved.")

## 8. Run Inference on a Single Image
Test the newly trained Qwen vision-language model on an uploaded image! (Upload `9_IM-2407-1001.dcm.png` to the Colab files section first).

In [ ]:
from PIL import Image
from medivlm.data.transforms import build_image_transform
import IPython.display as display

IMAGE_PATH = "/content/9_IM-2407-1001.dcm.png"

if not os.path.exists(IMAGE_PATH):
    print(f"⚠️ Please upload '{IMAGE_PATH}' into the Colab file browser first!")
else:
    print("🖼️ Processing Image...")
    img = Image.open(IMAGE_PATH).convert("RGB")
    display.display(img.resize((300, 300)))
    
    transform = build_image_transform(cfg.image_encoder.image_size, train=False)
    image_tensor = transform(img).unsqueeze(0).to(device)
    
    print("\n🔮 Generating Clinical Report using Qwen2.5-3B...")
    model.eval()
    with torch.no_grad():
        reports = model.generate(image_tensor, num_beams=4)
        
    print("\n" + "="*80)
    print("📄 GENERATED REPORT:")
    print("="*80)
    print(reports[0])
    print("="*80)